# M1 Notebook 22 — Numerical Optimization

**Status:** Runnable first edition

## Learning objectives

- Implement gradient descent and line search.
- Compare momentum, RMSProp, and Adam.
- Study learning-rate sensitivity.

In [ ]:
from srai_math.utils import environment_info,set_seed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
set_seed(42)
environment_info()
from srai_math.optimization import (
    adam,backtracking_line_search,gradient_descent,momentum_descent,rmsprop,
)


## Gradient descent

\[
x_{k+1}=x_k-\eta\nabla f(x_k).
\]


In [ ]:
f=lambda x: 100*(x[1]-x[0]**2)**2+(1-x[0])**2
g=lambda x: np.array([
    -400*x[0]*(x[1]-x[0]**2)-2*(1-x[0]),
    200*(x[1]-x[0]**2),
])
x0=np.array([-1.2,1.0])
results={}
for lr in [1e-4,5e-4,1e-3]:
    x,h,it=gradient_descent(f,g,x0,learning_rate=lr,max_iter=20000,tol=1e-12)
    results[lr]=(x,h,it)
pd.DataFrame([
    {"learning_rate":lr,"final_objective":h[-1],"iterations":it,"x1":x[0],"x2":x[1]}
    for lr,(x,h,it) in results.items()
])


In [ ]:
fig,ax=plt.subplots(figsize=(7,4))
for lr,(x,h,it) in results.items():
    ax.semilogy(h,label=f"lr={lr}")
ax.set_xlabel("Iteration"); ax.set_ylabel("Objective")
ax.set_title("Learning-Rate Sensitivity"); ax.legend()
plt.show()


## Backtracking line search

In [ ]:
direction=-g(x0)
alpha=backtracking_line_search(f,g,x0,direction)
{"accepted_step":alpha,"objective_before":f(x0),"objective_after":f(x0+alpha*direction)}


## Momentum, RMSProp, and Adam

In [ ]:
methods={
    "Momentum":momentum_descent(f,g,x0,learning_rate=.001,momentum=.9,max_iter=5000),
    "RMSProp":rmsprop(f,g,x0,learning_rate=.001,max_iter=5000),
    "Adam":adam(f,g,x0,learning_rate=.01,max_iter=5000),
}
pd.DataFrame([
    {"method":name,"objective":history[-1],"x1":x[0],"x2":x[1]}
    for name,(x,history) in methods.items()
])


In [ ]:
fig,ax=plt.subplots(figsize=(7,4))
for name,(x,h) in methods.items():
    ax.semilogy(h,label=name)
ax.set_xlabel("Iteration"); ax.set_ylabel("Objective")
ax.set_title("Optimizer Comparison"); ax.legend()
plt.show()


## Decision Intelligence case

Fit a small policy-response model by minimizing squared error.

In [ ]:
X=np.array([[1,0],[1,1],[1,2],[1,3]],dtype=float)
y=np.array([2.1,2.9,4.2,5.1])
loss=lambda b: float(np.mean((X@b-y)**2))
grad=lambda b: 2*X.T@(X@b-y)/len(y)
b,h,_=gradient_descent(loss,grad,[0,0],learning_rate=.1,max_iter=500)
{"coefficients":b,"loss":h[-1]}


## Engineering notes

Learning rates, scaling, conditioning, and stopping rules strongly affect convergence.

## Key insight

Numerical optimization converts derivatives into iterative algorithms whose behavior must be diagnosed, not assumed.